# Walk-Forward Backtesting & Ensemble Forecasting

Compare all candidates on aligned forecast origins and horizons. Multi-step lag features must consume prior predictions, never future actuals.


In [1]:
# PROJECT_ROOT_BOOTSTRAP: support kernels started from the repository root or notebooks/.
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Could not locate the project root containing src/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.data import load_demand, TARGETS
from src.backtesting import naive_forecast, seasonal_naive_forecast, recursive_predict
from src.evaluation import forecast_metrics, expanding_splits, horizon_band_metrics
from src.features import make_calendar_features, make_lag_features
from src.models.statsmodels_models import fit_candidates, forecast_frame
from src.models.prophet_models import prophet_frame, make_model, future_frame
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from pathlib import Path
from importlib.metadata import version
import json
import pandas as pd

demand = load_demand("data/raw/clientes_restaurantes.csv")
demand["Total demand"] = demand[list(TARGETS)].sum(axis=1, min_count=2)
development = demand.iloc[:-28].copy()  # Final 28 calendar dates remain untouched.
selection = json.loads(Path("artifacts/models/ml_hyperparameter_selection.json").read_text())["selected_parameters"]
fold_rows = []
prediction_rows = []
for target in [*TARGETS, "Total demand"]:
    y = development[target]
    for fold, (tr, te) in enumerate(expanding_splits(len(y), initial=280, horizon=28, step=28), start=1):
        train, test = y.iloc[tr].copy(), y.iloc[te].copy()
        if test.isna().any():
            print(target, fold, "excluded from scoring; unknown test dates:", test.index[test.isna()].tolist())
            continue
        missing_train = train.index[train.isna()]
        if len(missing_train):
            print(target, fold, "training-only interpolation dates:", missing_train.tolist())
        train = train.interpolate(limit_area="inside").dropna()
        candidates = {
            "naive": naive_forecast(train, test.index),
            "seasonal_naive_7": seasonal_naive_forecast(train, test.index),
        }
        intervals = {}
        for name, result in fit_candidates(train).items():
            if not isinstance(result, Exception):
                statistical_forecast = forecast_frame(result, len(test), test.index)
                candidate_name = f"statsmodels_{name}"
                candidates[candidate_name] = statistical_forecast["yhat"]
                intervals[candidate_name] = (statistical_forecast["yhat_lower"].to_numpy(), statistical_forecast["yhat_upper"].to_numpy())
        prophet = make_model(seasonality_mode="additive", yearly_seasonality=False)
        prophet.fit(prophet_frame(train))
        pforecast = prophet.predict(future_frame(prophet, len(test))).tail(len(test))
        candidates["prophet_additive"] = pd.Series(pforecast["yhat"].to_numpy(), index=test.index)
        intervals["prophet_additive"] = (pforecast["yhat_lower"].to_numpy(), pforecast["yhat_upper"].to_numpy())
        X = make_calendar_features(train.index).join(make_lag_features(train))
        valid = X.notna().all(axis=1)
        ml_parameters = selection[target]
        ml = make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingRegressor(random_state=42, **ml_parameters))
        ml.fit(X.loc[valid], train.loc[valid])
        def build_features(values):
            return make_calendar_features(values.index).join(make_lag_features(values))
        candidates["hist_gradient_boosting_recursive"] = recursive_predict(ml, train, test.index, build_features, target)
        for name, prediction in candidates.items():
            metrics = forecast_metrics(test, prediction, train)
            fold_rows.append({"restaurant": target, "fold": fold, "model": name, **metrics})
            bounds = intervals.get(name, (None, None))
            for step, (date, value) in enumerate(prediction.items(), start=1):
                lower = bounds[0][step-1] if bounds[0] is not None else None
                upper = bounds[1][step-1] if bounds[1] is not None else None
                prediction_rows.append({"date": date, "restaurant": target, "model": name, "horizon": step, "prediction": value, "lower": lower, "upper": upper})
            print(target, fold, name, horizon_band_metrics(test, prediction, train).to_dict("records"))

fold_results = pd.DataFrame(fold_rows)
summary = fold_results.groupby(["restaurant", "model"]).agg({"mae": ["mean", "std"], "rmse": ["mean", "std"], "wape": ["mean", "std"], "smape": ["mean", "std"], "mase": ["mean", "std"], "mean_error": ["mean", "std"]})
display(summary)
Path("reports").mkdir(exist_ok=True)
fold_results.to_csv("reports/fold_level_model_comparison.csv", index=False)
Path("artifacts/forecasts").mkdir(parents=True, exist_ok=True)
pd.DataFrame(prediction_rows).to_csv("artifacts/forecasts/walk_forward_forecasts.csv", index=False)
metadata = {"training_cutoffs": [str(development.index[tr.stop - 1].date()) for tr, _ in expanding_splits(len(development), initial=280, horizon=28, step=28)], "final_holdout_period": [str(demand.index[-28].date()), str(demand.index[-1].date())], "package_versions": {"pandas": version("pandas"), "scikit-learn": version("scikit-learn"), "statsmodels": version("statsmodels"), "prophet": version("prophet")}, "parameters": {"forecast_horizon_days": 28, "seasonal_naive_period": 7, "hist_gradient_boosting_random_state": 42, "hist_gradient_boosting_selected": selection, "statsmodels_search": "AutoReg maxlag 14; AR/MA p,q in 0..2, d in 0..1; weekly SARIMAX seasonal order search, period 7", "prophet_mode": "additive", "prophet_yearly_seasonality": False}, "validation_method": "aligned expanding-window forecasts with recursive HistGradientBoosting predictions"}
Path("artifacts/models/walk_forward_metadata.json").write_text(json.dumps(metadata, indent=2))


Chimi & Churri 1 training-only interpolation dates: [Timestamp('2016-04-05 00:00:00'), Timestamp('2016-09-17 00:00:00')]


Importing plotly failed. Interactive plots will not work.


18:26:57 - cmdstanpy - INFO - Chain [1] start processing


18:26:57 - cmdstanpy - INFO - Chain [1] done processing


Chimi & Churri 1 naive [{'horizon_start': 1, 'horizon_end': 7, 'mae': 30.857142857142858, 'rmse': 39.774363602702685, 'wape': 0.7941176470588235, 'smape': 1.0488202118454217, 'mean_error': -30.857142857142858, 'r2': -1.5117620374570673, 'mase': 1.9853876973839266}, {'horizon_start': 8, 'horizon_end': 14, 'mae': 29.285714285714285, 'rmse': 36.22745604411896, 'wape': 0.7976653696498055, 'smape': 1.1177933340681985, 'mean_error': -28.714285714285715, 'r2': -1.6898527689476324, 'mase': 1.884279990572708}, {'horizon_start': 15, 'horizon_end': 21, 'mae': 29.0, 'rmse': 33.413000036341714, 'wape': 0.7837837837837838, 'smape': 1.2063853808459224, 'mean_error': -29.0, 'r2': -3.053423236514523, 'mase': 1.8658967711524865}, {'horizon_start': 22, 'horizon_end': 28, 'mae': 37.714285714285715, 'rmse': 46.055866696251165, 'wape': 0.825, 'smape': 1.2042690458072856, 'mean_error': -37.714285714285715, 'r2': -2.035514018691589, 'mase': 2.4265849634692436}]
Chimi & Churri 1 seasonal_naive_7 [{'horizon_sta

18:27:01 - cmdstanpy - INFO - Chain [1] start processing


... (42 outputs left out)

18:28:01 - cmdstanpy - INFO - Chain [1] done processing


Total demand 6 naive [{'horizon_start': 1, 'horizon_end': 7, 'mae': 22.857142857142858, 'rmse': 32.96751647997075, 'wape': 0.367816091954023, 'smape': 0.3508256690268324, 'mean_error': -1.1428571428571428, 'r2': -0.001203188449390602, 'mase': 0.9171281453414942}, {'horizon_start': 8, 'horizon_end': 14, 'mae': 25.714285714285715, 'rmse': 30.085592185344417, 'wape': 0.3938730853391685, 'smape': 0.39363480600978745, 'mean_error': -4.285714285714286, 'r2': -0.0207125103562551, 'mase': 1.031769163509181}, {'horizon_start': 15, 'horizon_end': 21, 'mae': 33.857142857142854, 'rmse': 51.69553724866946, 'wape': 0.462890625, 'smape': 0.44761645031318226, 'mean_error': -12.142857142857142, 'r2': -0.05839610746500257, 'mase': 1.3584960652870883}, {'horizon_start': 22, 'horizon_end': 28, 'mae': 36.142857142857146, 'rmse': 41.052231538440324, 'wape': 0.4566787003610108, 'smape': 0.5126496175138755, 'mean_error': -18.142857142857142, 'r2': -0.24272385252069228, 'mase': 1.4502088798212378}]
Total deman

mae            \
                                                      mean       std   
restaurant     model                                                   
Assa Frão      hist_gradient_boosting_recursive  14.781985  4.105710   
               naive                             19.464286  3.493069   
               prophet_additive                  13.079049  3.120707   
               seasonal_naive_7                  18.750000  3.869029   
               statsmodels_ARIMA_AIC             18.926560  1.885547   
               statsmodels_ARMA_AIC              19.841049  1.810010   
               statsmodels_SARIMAX_weekly_AIC    12.153401  2.892579   
Chimi & Churri hist_gradient_boosting_recursive  12.455973  2.311099   
               naive                             22.208333  6.575442   
               prophet_additive                  11.673739  1.575361   
               seasonal_naive_7                  16.851190  4.331742   
               statsmodels_ARIMA_AIC             16.931636  1.536414   
               statsmodels_ARMA_AIC              14.290061  1.660051   
               statsmodels_SARIMAX_weekly_AIC    11.884964  1.977102   
Total demand   hist_gradient_boosting_recursive  22.148339  6.238060   
               naive                             35.507143  6.994230   
               prophet_additive                  17.896759  4.523920   
               seasonal_naive_7                  25.050000  4.412023   
               statsmodels_ARIMA_AIC             30.895081  2.478942   
               statsmodels_ARMA_AIC              21.114108  3.511131   
               statsmodels_SARIMAX_weekly_AIC    18.659615  4.242546   

                                                      rmse            \
                                                      mean       std   
restaurant     model                                                   
Assa Frão      hist_gradient_boosting_recursive  20.425898  4.873956   
               naive                             26.726687  3.564196   
               prophet_additive                  19.580690  5.019613   
               seasonal_naive_7                  26.641453  4.981771   
               statsmodels_ARIMA_AIC             24.132087  3.761698   
               statsmodels_ARMA_AIC              24.500650  3.547449   
               statsmodels_SARIMAX_weekly_AIC    19.685205  4.725329   
Chimi & Churri hist_gradient_boosting_recursive  15.918763  2.327701   
               naive                             28.216037  6.562166   
               prophet_additive                  14.697123  2.036072   
               seasonal_naive_7                  20.607658  4.918992   
               statsmodels_ARIMA_AIC             22.568110  2.930573   
               statsmodels_ARMA_AIC              18.086954  2.409013   
               statsmodels_SARIMAX_weekly_AIC    15.106449  2.358055   
Total demand   hist_gradient_boosting_recursive  27.559673  6.423353   
               naive                             46.148227  7.960147   
               prophet_additive                  25.030309  6.205662   
               seasonal_naive_7                  33.545448  6.874781   
               statsmodels_ARIMA_AIC             37.954388  3.749253   
               statsmodels_ARMA_AIC              27.561602  5.244862   
               statsmodels_SARIMAX_weekly_AIC    26.115551  5.573696   

                                                     wape               smape  \
                                                     mean       std      mean   
restaurant     model                                                            
Assa Frão      hist_gradient_boosting_recursive  0.436215  0.122875  0.436304   
               naive                             0.568289  0.064498  0.587756   
               prophet_additive                  0.383420  0.082649  0.389893   
               seasonal_naive_7                  0.548611  0.092413  0.504755   
               statsmodels_ARI

1108